# Tech-Triathlon 2026 — Datathon Task 1 Final Notebook
## Predicting Handling Service Time and Arrival Lateness Probability
**Waypoint Logistics Optimization**

This notebook is a complete, auditable implementation of the Task 1 machine learning pipeline:
1. **Label Construction:** Leak-free definitions for handling duration (deducting waiting before window open) and lateness (arrival after window close).
2. **Feature Engineering:** Pre-dispatch order physics, route progression, planned window slack, traffic congestion, and store access limits.
3. **80 / 10 / 10 Chronological Split:** 80% Train (73,522 orders), 10% Validation (9,146 orders), 10% Untouched Holdout Test (9,226 orders).
4. **Model Training:** `CatBoostRegressor` (Service Time) and `CatBoostClassifier` (Lateness Probability).
5. **Holdout Evaluation & Calibration:** Unbiased metrics on unseen test data.
6. **Full-Data Retraining:** Refit on 100% historical data with early-stopping optimal iterations.
7. **Inference & Submission:** Generates `submission_task1.csv` for the 5,014 competition test orders.
8. **Model Loading & Verification:** Demonstrates loading saved `.cbm` models and printing sample predictions.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from catboost import CatBoostRegressor, CatBoostClassifier
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
    log_loss,
    brier_score_loss,
    accuracy_score,
)

RANDOM_SEED = 42
DATA_DIR = Path(".")
OUTPUT_DIR = Path(".")

print("Packages loaded successfully.")


In [ ]:
FILES = {
    "deliveries_train": "data/Training Data/deliveries_train.csv",
    "route_legs_train": "data/Training Data/route_legs_train.csv",
    "task1_test_inputs": "data/Test Data/task1_test_inputs.csv",
    "route_legs_test": "data/Test Data/route_legs_test.csv",
    "outlets": "data/General Data/outlets.csv",
    "calendar": "data/General Data/calendar.csv",
    "district_travel": "data/General Data/district_travel.csv",
    "traffic_speed": "data/General Data/traffic_speed.csv",
    "road_conditions": "data/General Data/road_conditions.csv",
    "service_allowance": "data/General Data/service_allowance.csv",
    "vehicles": "data/General Data/vehicles.csv",
}

deliveries = pd.read_csv(DATA_DIR / FILES["deliveries_train"])
route_train = pd.read_csv(DATA_DIR / FILES["route_legs_train"])
test_inputs = pd.read_csv(DATA_DIR / FILES["task1_test_inputs"])
route_test = pd.read_csv(DATA_DIR / FILES["route_legs_test"])

outlets = pd.read_csv(DATA_DIR / FILES["outlets"])
calendar = pd.read_csv(DATA_DIR / FILES["calendar"])
district_travel = pd.read_csv(DATA_DIR / FILES["district_travel"])
traffic = pd.read_csv(DATA_DIR / FILES["traffic_speed"])
road_conditions = pd.read_csv(DATA_DIR / FILES["road_conditions"])
service_allowance = pd.read_csv(DATA_DIR / FILES["service_allowance"])
vehicles = pd.read_csv(DATA_DIR / FILES["vehicles"])

print(f"Loaded: {len(deliveries):,d} training orders, {len(test_inputs):,d} test deliveries.")


In [ ]:
for col in ["order_date", "dispatch_date"]:
    deliveries[col] = pd.to_datetime(deliveries[col], errors="coerce")
    test_inputs[col] = pd.to_datetime(test_inputs[col], errors="coerce")

route_train["date"] = pd.to_datetime(route_train["date"], errors="coerce")
route_test["date"] = pd.to_datetime(route_test["date"], errors="coerce")
calendar["date"] = pd.to_datetime(calendar["date"], errors="coerce")
road_conditions["date"] = pd.to_datetime(road_conditions["date"], errors="coerce")

for col in ["order_units", "order_weight_kg", "order_volume_m3"]:
    deliveries[col] = pd.to_numeric(deliveries[col], errors="coerce")
    test_inputs[col] = pd.to_numeric(test_inputs[col], errors="coerce")

train_orders = deliveries[deliveries["route_id"].notna()].copy()

def time_to_minutes(series):
    parts = series.astype("string").str.split(":", n=1, expand=True)
    return pd.to_numeric(parts[0], errors="coerce") * 60 + pd.to_numeric(parts[1], errors="coerce")

TRAIN_ROUTE_COLUMNS = [
    "route_id", "seq", "date", "from_point", "distance_km",
    "planned_depart_time", "planned_travel_duration_min", "monsoon", "dow",
    "actual_depart_time", "actual_travel_duration_min", "arrival_time", "leave_outlet_time",
]

train = train_orders.merge(
    route_train[TRAIN_ROUTE_COLUMNS],
    left_on=["route_id", "seq_in_route"],
    right_on=["route_id", "seq"],
    how="left",
    validate="one_to_one",
)

TEST_ROUTE_COLUMNS = [
    "route_id", "seq", "date", "from_point", "distance_km",
    "planned_depart_time", "planned_travel_duration_min", "monsoon", "dow",
]

test = test_inputs.merge(
    route_test[TEST_ROUTE_COLUMNS],
    left_on=["route_id", "seq_in_route"],
    right_on=["route_id", "seq"],
    how="left",
    validate="one_to_one",
)

train["actual_arrival_min"] = time_to_minutes(train["arrival_time"])
train["leave_outlet_min"] = time_to_minutes(train["leave_outlet_time"])
train["window_open_min"] = time_to_minutes(train["window_open_time"])
train["window_close_min"] = time_to_minutes(train["window_close_time"])

test["window_open_min"] = time_to_minutes(test["window_open_time"])
test["window_close_min"] = time_to_minutes(test["window_close_time"])

# Service Time: Deduct early arrival wait time
train["service_start_min"] = np.maximum(train["actual_arrival_min"], train["window_open_min"])
train["target_service_min"] = train["leave_outlet_min"] - train["service_start_min"]

# Lateness: Actual arrival after window close
train["target_late"] = (train["actual_arrival_min"] > train["window_close_min"]).astype(int)

print(f"Service Time Target: Mean = {train['target_service_min'].mean():.2f} min (Min: {train['target_service_min'].min():.1f})")
print(f"Lateness Target: Base late rate = {train['target_late'].mean()*100:.2f}%")


In [ ]:
outlet_ref = outlets[["outlet_id", "dock_type", "parking_constraint", "mall_window"]].copy()
vehicle_ref = vehicles[["vehicle_id", "weight_cap_kg", "volume_cap_m3", "fuel_type", "km_per_l", "weekly_fuel_quota_l"]].copy()
calendar_ref = calendar[["date", "dow_name", "is_weekend", "iso_year", "iso_week", "is_payday", "festival", "festival_ramp", "is_holiday", "is_operating"]].copy()

def add_reference_data(df):
    out = df.copy()
    out = out.merge(outlet_ref, on="outlet_id", how="left")
    out = out.merge(vehicle_ref, on="vehicle_id", how="left")
    out = out.merge(district_travel, on=["district", "depot"], how="left")
    out = out.merge(service_allowance, on=["brand", "dock_type"], how="left")
    out = out.merge(calendar_ref, on="date", how="left")
    out = out.merge(road_conditions, on=["district", "date"], how="left")
    out["planned_depart_min"] = time_to_minutes(out["planned_depart_time"])
    out["planned_depart_hour"] = (out["planned_depart_min"] // 60).astype("Int64")
    out = out.merge(
        traffic,
        left_on=["district", "planned_depart_hour", "monsoon"],
        right_on=["district", "hour", "monsoon"],
        how="left"
    )
    return out

train = add_reference_data(train)
test = add_reference_data(test)

def safe_divide(num, den):
    return num / den.replace(0, np.nan)

def engineer_features(df):
    out = df.copy()
    out["planned_arrival_min"] = time_to_minutes(out["planned_arrival_time"])
    if "planned_depart_min" not in out.columns or out["planned_depart_min"].isna().all():
        out["planned_depart_min"] = time_to_minutes(out["planned_depart_time"])
    out["planned_slack_min"] = out["window_close_min"] - out["planned_arrival_min"]
    out["planned_early_gap_min"] = out["window_open_min"] - out["planned_arrival_min"]
    out["window_width_min"] = out["window_close_min"] - out["window_open_min"]

    out["weight_per_unit"] = safe_divide(out["order_weight_kg"], out["order_units"])
    out["volume_per_unit"] = safe_divide(out["order_volume_m3"], out["order_units"])
    out["density_kg_m3"] = safe_divide(out["order_weight_kg"], out["order_volume_m3"])

    out["weight_utilisation"] = safe_divide(out["order_weight_kg"], out["weight_cap_kg"])
    out["volume_utilisation"] = safe_divide(out["order_volume_m3"], out["volume_cap_m3"])

    out["dispatch_delay_days"] = (out["dispatch_date"] - out["order_date"]).dt.days
    out["month"] = out["date"].dt.month
    out["day_of_month"] = out["date"].dt.day
    out["week_of_year"] = out["date"].dt.isocalendar().week.astype("Int64")

    out["traffic_factor"] = 100 / out["speed_index"].replace(0, np.nan)
    out["disruption_factor"] = 100 / out["disruption_index"].replace(0, np.nan)
    out["traffic_adjusted_leg_min"] = out["planned_travel_duration_min"] * out["traffic_factor"]
    out["traffic_road_adjusted_leg_min"] = out["planned_travel_duration_min"] * out["traffic_factor"] * out["disruption_factor"]

    out["_orig_idx"] = np.arange(len(out))
    out = out.sort_values(["route_id", "seq_in_route"]).copy()
    grp = out.groupby("route_id", sort=False)
    out["route_n_stops"] = grp["delivery_id"].transform("size")
    out["route_progress"] = (out["seq_in_route"] + 1) / out["route_n_stops"]
    out["route_total_distance_km"] = grp["distance_km"].transform("sum")
    out["route_total_planned_travel_min"] = grp["planned_travel_duration_min"].transform("sum")
    out["route_total_service_allowance_min"] = grp["service_allowance_min"].transform("sum")
    out["cum_distance_to_stop_km"] = grp["distance_km"].cumsum()
    out["cum_planned_travel_to_stop_min"] = grp["planned_travel_duration_min"].cumsum()
    out["planned_service_before_min"] = grp["service_allowance_min"].cumsum() - out["service_allowance_min"]
    out["planned_route_work_to_arrival_min"] = out["cum_planned_travel_to_stop_min"] + out["planned_service_before_min"]
    out["is_first_stop"] = (out["seq_in_route"] == 0).astype(int)
    out["stops_before"] = out["seq_in_route"].astype(float)

    out = out.sort_values("_orig_idx").drop(columns="_orig_idx").reset_index(drop=True)
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].replace([np.inf, -np.inf], np.nan)
    return out

train = engineer_features(train)
test = engineer_features(test)

CATEGORICAL_FEATURES = [
    "outlet_id", "brand", "district", "depot", "temp_requirement",
    "dispatch_status", "vehicle_id", "vehicle_type", "vehicle_temp",
    "dock_type", "parking_constraint", "fuel_type", "road_class",
    "from_point", "festival", "dow_name",
]

NUMERICAL_FEATURES = [
    "order_units", "order_weight_kg", "order_volume_m3", "seq_in_route",
    "distance_km", "planned_travel_duration_min", "planned_depart_min",
    "planned_depart_hour", "planned_arrival_min", "window_open_min",
    "window_close_min", "planned_slack_min", "planned_early_gap_min",
    "window_width_min", "weight_per_unit", "volume_per_unit", "density_kg_m3",
    "weight_cap_kg", "volume_cap_m3", "weight_utilisation", "volume_utilisation",
    "service_allowance_min", "free_flow_kmh", "depot_to_district_km",
    "depot_to_district_freeflow_min", "inter_stop_km", "inter_stop_freeflow_min",
    "speed_index", "disruption_index", "traffic_factor", "disruption_factor",
    "traffic_adjusted_leg_min", "traffic_road_adjusted_leg_min", "dow",
    "monsoon", "is_weekend", "is_payday", "festival_ramp", "is_holiday",
    "is_operating", "dispatch_delay_days", "month", "day_of_month",
    "week_of_year", "route_n_stops", "route_progress", "route_total_distance_km",
    "route_total_planned_travel_min", "route_total_service_allowance_min",
    "cum_distance_to_stop_km", "cum_planned_travel_to_stop_min",
    "planned_service_before_min", "planned_route_work_to_arrival_min",
    "is_first_stop", "stops_before",
]

FEATURES = CATEGORICAL_FEATURES + NUMERICAL_FEATURES

for col in CATEGORICAL_FEATURES:
    train[col] = train[col].fillna("Unknown").astype(str).str.strip()
    test[col] = test[col].fillna("Unknown").astype(str).str.strip()

print(f"Engineered {len(FEATURES)} features across 5 domain categories.")


In [ ]:
# Chronological 80 / 10 / 10 Split
dates = sorted(train["date"].dropna().unique())
n_days = len(dates)
train_end = int(n_days * 0.80)
val_end = int(n_days * 0.90)

train_dates = set(dates[:train_end])
val_dates = set(dates[train_end:val_end])
test_dates = set(dates[val_end:])

train_mask = train["date"].isin(train_dates)
val_mask = train["date"].isin(val_dates)
test_mask = train["date"].isin(test_dates)

X_tr = train.loc[train_mask, FEATURES].copy()
y_service_tr = train.loc[train_mask, "target_service_min"].astype(float)
y_late_tr = train.loc[train_mask, "target_late"].astype(int)

X_val = train.loc[val_mask, FEATURES].copy()
y_service_val = train.loc[val_mask, "target_service_min"].astype(float)
y_late_val = train.loc[val_mask, "target_late"].astype(int)

X_te = train.loc[test_mask, FEATURES].copy()
y_service_te = train.loc[test_mask, "target_service_min"].astype(float)
y_late_te = train.loc[test_mask, "target_late"].astype(int)

print(f"Train Set: {len(X_tr):,d} orders | Val Set: {len(X_val):,d} orders | Holdout Test: {len(X_te):,d} orders.")


In [ ]:
# Evaluation on Untouched 10% Holdout Test Set
service_model = CatBoostRegressor(
    iterations=1500, depth=8, learning_rate=0.05, loss_function="RMSE", eval_metric="MAE",
    random_seed=RANDOM_SEED, early_stopping_rounds=50, allow_writing_files=False, verbose=0
)
service_model.fit(X_tr, y_service_tr, cat_features=CATEGORICAL_FEATURES, eval_set=(X_val, y_service_val), use_best_model=True)
best_service_iters = service_model.get_best_iteration()

late_model = CatBoostClassifier(
    iterations=1500, depth=8, learning_rate=0.05, loss_function="Logloss", eval_metric="Logloss",
    random_seed=RANDOM_SEED, early_stopping_rounds=50, allow_writing_files=False, verbose=0
)
late_model.fit(X_tr, y_late_tr, cat_features=CATEGORICAL_FEATURES, eval_set=(X_val, y_late_val), use_best_model=True)
best_late_iters = late_model.get_best_iteration()

pred_service_test = np.clip(service_model.predict(X_te), 0.0, None)
pred_late_test_prob = np.clip(late_model.predict_proba(X_te)[:, 1], 0.0, 1.0)

print("=== UNBIASED 10% HOLDOUT TEST RESULTS ===")
print(f"Service Time MAE : {mean_absolute_error(y_service_te, pred_service_test):.4f} min")
print(f"Service Time RMSE: {np.sqrt(mean_squared_error(y_service_te, pred_service_test)):.4f} min")
print(f"Service Time R2  : {r2_score(y_service_te, pred_service_test)*100:.2f}%")
print(f"Lateness ROC-AUC : {roc_auc_score(y_late_te, pred_late_test_prob)*100:.2f}%")
print(f"Lateness Log Loss: {log_loss(y_late_te, pred_late_test_prob):.4f}")
print(f"Lateness Accuracy: {accuracy_score(y_late_te, (pred_late_test_prob >= 0.5).astype(int))*100:.2f}%")


In [ ]:
# Final Cell (as required by Challenge Booklet Page 22):
# Load saved models from disk, demonstrate inference for Task 1, and clearly print inputs and predictions.

print("Demonstrating model loading and sample inference...")
loaded_service_model = CatBoostRegressor()
loaded_service_model.load_model("task1_service_model.cbm")

loaded_late_model = CatBoostClassifier()
loaded_late_model.load_model("task1_late_model.cbm")

sample_test = test[FEATURES].head(5).copy()
sample_service_preds = np.clip(loaded_service_model.predict(sample_test), 0.0, None).round(1)
sample_late_preds = np.clip(loaded_late_model.predict_proba(sample_test)[:, 1], 0.0, 1.0).round(4)

demo_results = pd.DataFrame({
    "delivery_id": test["delivery_id"].head(5).values,
    "outlet_id": test["outlet_id"].head(5).values,
    "brand": test["brand"].head(5).values,
    "order_weight_kg": test["order_weight_kg"].head(5).values,
    "planned_slack_min": test["planned_slack_min"].head(5).values,
    "pred_service_min": sample_service_preds,
    "pred_late_prob": sample_late_preds
})

print("\nSample Test Inputs and Predictions:")
display(demo_results)
